# Choose reference portfolio return 

In [ ]:
from pprdyn1 import *
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
import pickle
import numpy as np
import pandas as pd
from performance_eval import avgperformance

config = {'policytype': 0, 'num_episodes': 1000}
settings = {'settingID': 18, 'for_RL': 0}
env = pprdyn1(settings)
summary = avgperformance(env, config, collect_data=True)
# average return using value iteraiton policy for each timestep.
summary['portfolioreturn'].mean(axis=0)
# Recommendation: average of the average returns is about *1.85*. Use this value as R_ref. 

policy type: value iteration
Episode 1000 done
MC log_negV = -6.590662
Average reward over 1000 episodes: -9.3339, certrainty equivalent: 1.3673


array([2.22334062, 2.01649829, 1.85343344, 1.77733091, 1.75704407,
       1.76121069, 1.78069268, 1.76588749, 1.75929858, 1.75398583])

In [1]:
# PPO training script
from PPO import *
from call_paramset import call_paramset, call_env
import pandas as pd
import random
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

if __name__ == "__main__":
    paramid = 6 # test id = 4
    iteration_num = 30
    hyperparameterization_set_filename = './hyperparamsets/PPOhyperparamsets.csv'
    paramdflist = call_paramset(hyperparameterization_set_filename,paramid)
    tuneset = 1
    best_scores = []
    seeds = []
    for paramdf in paramdflist: 
        for iteration in range(iteration_num):
            print('-------------------------------------------------------\n--------------------------------------------------------')
            seed = paramdf['seed'] # make sure iteration_num matches with the number of seeds if seeds are specified
            if seed == 'random':
                seednum = random.randint(0,1000000)
            else:
                seednum = int(seed)
            os.environ["PYTHONHASHSEED"] = str(seednum)
            random.seed(seednum)
            np.random.seed(seednum)
            torch.manual_seed(seednum)
            # environment configuration
            env = call_env(paramdf)
            meta = {'paramid': paramid, 'iteration': iteration, 'seed': seednum}
            agent = PPO(env, paramdf,meta)
            _, scores = agent.train()
            best_scores.append(np.max(scores))
            seeds.append(seednum)
            

            print(f"Tuneset {tuneset}, Iteration {iteration}: {scores}")
        tuneset += 1
    seedsNscores = pd.DataFrame({'seed': seeds, 'best_score': best_scores})
    print(seedsNscores.sort_values(by='best_score', ascending=False))

-------------------------------------------------------
--------------------------------------------------------


Using cpu device
paramID: 6, iteration: 0, seed: 632857
env: pprdyn1
device: cpu
state size: 8, action size: 231
actor: hidden num: 3, hidden size: [32, 64, 32]
critic: hidden num: 2, hidden size: [256, 256]
actor lr: 0.0005, actor lr decay rate: 1.0, actor min lr: -inf
critic lr: 0.0003, critic lr decay rate: 1.0, critic min lr: -inf
standardize: True
gamma: 0.999, gae_lambda: 0.95
advantage normalization: True
KL stopping: True, target KL: 0.02
c1: 0.5, c2: 0.001, entropy_loss_included: True, policy_clip: 0.2
rollout length: 512, minibatch size: 64, n_epochs: 12
evaluation interval: 500, performance sampleN: 1000, parallel testing: False, deterministic eval: False
max steps: 10, episodenum: 30000
Episode 500
serial calc_performance called
Episode 500, Learning Rate: A0.0005/C0.0003 Avg Performance: 1.55
-----------------------------------
Episode 1000
serial calc_performance called
Episode 1000, Learning Rate: A0.0005/C0.0003 Avg Performance: 1.57
-----------------------------------
